# Лекција 08 - Образац мулти-агент дизајна


## Подешавање


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Зашто мултиагентски системи?

Задаће из стварног света као што је планирање путовања подразумевају много различитих врста експертизе — логистику, локално познавање, буџетирање и још много тога. Један агент који покушава све да реши постаје брзо незгодан.

Мултиагентски системи ово решавају кроз **специјализацију**: сваки агент се фокусира на једну област експертизе, производећи резултате бољег квалитета од генералисте. Они такође побољшавају **скалабилност** — можете додати нове агенте (нпр. специјалисту за летове, критичара ресторана) без преправљања постојећег радног тока. Агенти се повезују кроз структуриран процес, преносећи контекст један другом.


## Креирање специјализованих агената


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Прављење секвенцијалног радног тока

`WorkflowBuilder` вам омогућава да повежете агенте у усмерени граф. Овде правимо једноставан двостепени процес: **TravelPlanner** саставља план пута, а онда га **TravelConcierge** прегледа и унапређује.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавање још агената у ток рада

Једна од највећих предности мулти-агентног обрасца је колико је лако проширити га. Испод додајемо агента **BudgetReviewer** који проверава план у складу са буџетом путника, означава ставке које би могле премашити лимит трошкова и предлаже алтернативе које штеде новац. Ток рада сада покреће три агента узастопно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

1. **Креирате специјализоване агенте** — сваки са фокусираним улогом (планирање, консијерџ, преглед буџета).
2. **Повежете агенте у секвенцијални радни ток** користећи `WorkflowBuilder` и `add_edge`.
3. **Стримујете излаз** из вишеагентског процеса, пратећи који агент говори.
4. **Проширите радни ток** додавањем нових агената у ланац без модификовања постојећих.

Дизајн шаблона са више агената држи сваки агент једноставним, док производи богатије, темељније прегледане резултате него што би то могао постићи било који појединачни агент самостално.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге аутоматског превођења [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде прецизан, молимо да имате у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетом. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произлазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
